In [15]:
# ============================================================================
# Cell 1 - Configuration and environment
# ============================================================================
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q -U transformers accelerate seqeval

import sys, json, math, time, random, gc, collections, statistics, unicodedata
import numpy as np
import torch

import transformers
print(f"transformers {transformers.__version__} | torch {torch.__version__}")

NOTEBOOK_VERSION = "2026-09-25.1"
print("=" * 78)
print(f"sinfundMultitaskLayout.ipynb   revision {NOTEBOOK_VERSION}")
print("=" * 78)

# ---------------------------------------------------------------- CONFIG ----
CFG = dict(
    smoke_test        = True,   # tiny end-to-end pass; metrics meaningless by design

    # --- arm -------------------------------------------------------------
    # "lilt-infoxlm"  : SCUT-DLVCLab/lilt-infoxlm-base. Its tokenizer IS layoutxlm's
    #                   (verified: tokenizer_config.json name_or_path), i.e. XLM-R ->
    #                   covers Sinhala. This is the paper's architecture family.
    # "lilt-xlmr"     : nielsr/lilt-xlm-roberta-base - closest to the paper's stated
    #                   "LiLT + XLM-R" wording.
    # "xlmr-textonly" : ablation with NO layout stream. Isolates how much the layout
    #                   modality actually contributes - never reported in the paper.
    # NOTE: layoutxlm is deliberately absent. transformers implements it via
    # LayoutLMv2, whose visual backbone hard-requires detectron2 (fragile to build on
    # Kaggle). It belongs in its own run, not as a dependency of this one.
    arm               = "lilt-infoxlm",

    # --- task ------------------------------------------------------------
    do_ser            = True,
    do_re             = True,

    # --- sequence --------------------------------------------------------
    # Measured on this dataset: words/form median 213, max 366. After subword
    # expansion that overflows 512, so documents are encoded in overlapping windows
    # and entity representations are pooled across them.
    max_length        = 512,
    window_stride     = 128,

    # --- optimisation ----------------------------------------------------
    ser_epochs        = 40,
    ser_lr            = 5e-5,
    ser_batch         = 2,
    re_epochs         = 60,
    re_lr             = 1e-4,
    re_proj_dim       = 128,
    re_dropout        = 0.1,
    # RE candidates are Q x A per document: 37,709 train candidates vs 1,604 gold
    # positives = 4.3%. Unweighted BCE would collapse to all-negative.
    re_pos_weight     = 0.0,     # 0 => computed from the data in Cell 8

    weight_decay      = 0.01,
    warmup_ratio      = 0.1,
    max_grad_norm     = 1.0,
    seed              = 3407,

    # --- data ------------------------------------------------------------
    # Of 2,010 unique links, 1,768 are stored (question,answer) and 221 reversed.
    # Normalising by entity LABEL rather than tuple order removes that 11%
    # inconsistency. Set False to measure what it costs (roadmap Step 3).
    normalize_link_direction = True,
    assert_dataset_identity  = True,

    val_fraction      = 0.15,    # carved out of the 80 train forms; test split untouched
    train_hours_budget = 5.0,
)
if CFG["smoke_test"]:
    CFG.update(ser_epochs=2, re_epochs=2, train_hours_budget=0.4)
    print(">>> SMOKE TEST MODE: results are NOT meaningful\n")

ARMS = {
    "lilt-infoxlm":  dict(repo="SCUT-DLVCLab/lilt-infoxlm-base",  uses_bbox=True),
    "lilt-xlmr":     dict(repo="nielsr/lilt-xlm-roberta-base",    uses_bbox=True),
    "xlmr-textonly": dict(repo="xlm-roberta-base",                uses_bbox=False),
}
assert CFG["arm"] in ARMS, f"arm must be one of {list(ARMS)}"
ARM = ARMS[CFG["arm"]]
print(f"\narm: {CFG['arm']}  ->  {ARM['repo']}  (layout stream: {ARM['uses_bbox']})")

# ------------------------------------------------------- GPU: fail fast -----
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. Kaggle -> Settings -> Accelerator -> 'GPU T4 x2'. Not TPU.")

cap = torch.cuda.get_device_capability(0)
ARCH_LIST, DEV_ARCH = torch.cuda.get_arch_list(), f"sm_{cap[0]}{cap[1]}"
print(f"\nGPU        : {torch.cuda.get_device_name(0)}  (CC {cap[0]}.{cap[1]})")
print(f"VRAM       : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
print(f"torch archs: {ARCH_LIST}")
if DEV_ARCH not in ARCH_LIST:
    raise RuntimeError(
        f"This PyTorch has NO kernels for {DEV_ARCH} ({torch.cuda.get_device_name(0)}).\n"
        f"FIX: Accelerator -> 'GPU T4 x2' (sm_75). Never P100 (sm_60).")

# T4 is sm_75: bf16 needs Ampere+ (CC>=8). torch.cuda.is_bf16_supported() reports True
# on T4 via EMULATION, which is slow and misleading - test the capability directly.
HAS_NATIVE_BF16 = cap[0] >= 8
AMP_DTYPE = torch.bfloat16 if HAS_NATIVE_BF16 else torch.float16
print(f"native bf16: {HAS_NATIVE_BF16}  =>  AMP dtype {AMP_DTYPE}")
# FlashAttention-2 requires Ampere+; Turing only has the separate flash-attention-turing
# fork. sdpa is the correct, always-available choice here.
ATTN_IMPL = "sdpa"
print(f"attn impl  : {ATTN_IMPL}")
try:
    _p = torch.randn(64, 64, device="cuda", dtype=AMP_DTYPE)
    _ = (_p @ _p).sum().item(); del _p; torch.cuda.synchronize()
    print(f"CUDA probe : OK ({AMP_DTYPE} matmul ran on device)")
except Exception as e:
    raise RuntimeError(f"A trivial {AMP_DTYPE} matmul failed: {e}")

SEED = CFG["seed"]
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

IN_KAGGLE  = os.path.isdir("/kaggle")
OUTPUT_DIR = "/kaggle/working" if IN_KAGGLE else "."
RUN_DIR    = os.path.join(OUTPUT_DIR, "sinfund_run")
os.makedirs(RUN_DIR, exist_ok=True)
print(f"\nrun dir: {RUN_DIR}")
print(json.dumps(CFG, indent=2))


  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
transformers 5.0.0 | torch 2.10.0+cu128
sinfundMultitaskLayout.ipynb   revision 2026-09-25.1
>>> SMOKE TEST MODE: results are NOT meaningful


arm: lilt-infoxlm  ->  SCUT-DLVCLab/lilt-infoxlm-base  (layout stream: True)

GPU        : Tesla T4  (CC 7.5)
VRAM       : 14.6 GB
torch archs: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
native bf16: False  =>  AMP dtype torch.float16
attn impl  : sdpa
CUDA probe : OK (torch.float16 matmul ran on device)

run dir: /kaggle/working/sinfund_run
{
  "

# SinFUND — Semantic Entity Recognition + Relation Extraction

Beats-or-reproduces the published SinFUND baseline: **SER F1 0.7560 / RE F1 0.3610**.

## Prerequisites
1. **Accelerator: `GPU T4 x2`.** Never P100 — no `sm_60` kernels, and 8-bit bitsandbytes needs CC>=7.5.
2. **Internet: ON** (downloads the base model from the Hub).
3. **Add Input -> Datasets -> the SinFUND upload.** Cell 2 finds it by *structure*, so the
   dataset's name does not matter — only that it contains
   `training_data/{annotations,images}` and `testing_data/{annotations,images}`.
4. No `HF_TOKEN` needed: every model used here is public.

## What this run must produce
| Check | Expected |
|---|---|
| Cell 1 revision banner | the revision you intended to run |
| Cell 2 identity assert | train 1,604 / test 406 relations |
| Cell 3 fertility | Sinhala tokens-per-word (settles an open question) |
| Cell 6 sanity tier | **exactly 1.0000** — else nothing else counts |
| Cell 7 SER | per-class F1, incl. the 3.6% `header` class |
| Cell 8 RE | F1 with the P/R split (published: P 0.2809 / R 0.5051) |


In [16]:
# ============================================================================
# Cell 2 - Load SinFUND, normalise link direction, assert dataset identity
# ============================================================================

# Find the dataset by STRUCTURE, not by name. A real Kaggle run in a sibling track
# crashed because it searched for a directory literally named "SinFund" while the
# upload had mounted at ".../sinfund" with training_data/ directly inside.
def find_sinfund(roots=("/kaggle/input", ".", "..")):
    need = [("training_data","annotations"),("training_data","images"),
            ("testing_data","annotations"),("testing_data","images")]
    hits = []
    for root in roots:
        if not os.path.isdir(root): continue
        for dirpath, dirnames, _ in os.walk(root):
            if all(os.path.isdir(os.path.join(dirpath,a,b)) for a,b in need):
                hits.append(dirpath)
            if dirpath.count(os.sep) - root.count(os.sep) > 6:
                dirnames[:] = []
    return hits

hits = find_sinfund()
if not hits:
    print("Could not find SinFUND. Directory tree under /kaggle/input:")
    for dirpath, dirnames, filenames in os.walk("/kaggle/input"):
        d = dirpath.count(os.sep) - "/kaggle/input".count(os.sep)
        if d <= 3: print("   " * d + os.path.basename(dirpath) + "/")
    raise FileNotFoundError(
        "No directory containing training_data/{annotations,images} + "
        "testing_data/{annotations,images}. Attach the SinFUND dataset.")
DATA_ROOT = hits[0]
print(f"SinFUND found at: {DATA_ROOT}")
if len(hits) > 1: print(f"  (note: {len(hits)} candidates, using the first)")

LABELS = ["other","question","answer","header"]

def norm_box(b, W, H):
    # LiLT/LayoutLM expect a 0-1000 grid. A sibling notebook had a real bug where one
    # stream clipped raw pixel boxes to [0,1000] instead of normalising them.
    x0,y0,x1,y1 = b
    x0,x1 = sorted((x0,x1)); y0,y1 = sorted((y0,y1))
    f = lambda v,m: max(0, min(1000, int(1000*v/max(1,m))))
    return [f(x0,W), f(y0,H), f(x1,W), f(y1,H)]

import struct
def png_size(p):
    with open(p,"rb") as fh: head = fh.read(24)
    return struct.unpack(">II", head[16:24])

def load_split(split):
    ann_dir = os.path.join(DATA_ROOT, split, "annotations")
    img_dir = os.path.join(DATA_ROOT, split, "images")
    docs = []
    stats = collections.Counter()
    raw_entities = raw_links = 0     # counted BEFORE any modelling filter, for the identity assert

    for fn in sorted(os.listdir(ann_dir)):
        if not fn.endswith(".json"): continue
        rec = json.load(open(os.path.join(ann_dir, fn), encoding="utf-8"))["form"]
        stem = os.path.splitext(fn)[0]
        img_path = None
        for ext in (".png",".jpg",".jpeg"):
            p = os.path.join(img_dir, stem+ext)
            if os.path.exists(p): img_path = p; break
        if img_path is None:
            stats["missing_image"] += 1; continue
        try: W,H = png_size(img_path)
        except Exception: W,H = 1000,1000

        raw_entities += len(rec)
        raw_links += len({(a,b) for e in rec for (a,b) in
                          [l for l in e.get("linking",[]) if isinstance(l,(list,tuple)) and len(l)==2]})

        ents = []
        for e in rec:
            words = [w for w in e.get("words",[]) if str(w.get("text","")).strip()]
            box = norm_box(e["box"], W, H)
            is_blank = not words
            if is_blank:
                # 65 train entities are ALL `answer`, with whitespace-only text, a valid
                # box, and exactly one link: these are form fields the user left UNFILLED.
                # Dropping them would lose 68 relations and teach the model nothing about
                # empty answers - which, for "which questions did the user actually fill
                # in?", is precisely the case that matters. Keep them with a single
                # sentinel word so the layout stream still sees the box and the link
                # survives. BLANK_SURFACE is set in Cell 3 once the tokenizer is known.
                stats["blank_field_kept"] += 1
                words = [dict(text=None, box=box, blank=True)]
            else:
                words = [dict(text=str(w["text"]).strip(), box=norm_box(w["box"], W, H),
                              blank=False) for w in words]
            ents.append(dict(id=e["id"], label=e["label"], text=e.get("text",""),
                             box=box, words=words, is_blank=is_blank,
                             raw_linking=e.get("linking", [])))
        by_id = {e["id"]: e for e in ents}

        # Links are stored BIDIRECTIONALLY (each relation appears on both entities), and
        # 11% of the unique tuples are stored (answer,question) rather than
        # (question,answer). Dedup on the tuple, then orient by LABEL.
        uniq, pairs = set(), set()
        for e in ents:
            for link in e["raw_linking"]:
                if not (isinstance(link,(list,tuple)) and len(link)==2): continue
                a,b = link
                if a not in by_id or b not in by_id: stats["link_dangling"] += 1; continue
                if (a,b) in uniq: continue
                uniq.add((a,b))
                la, lb = by_id[a]["label"], by_id[b]["label"]
                if CFG["normalize_link_direction"]:
                    if   la=="question" and lb=="answer": pairs.add((a,b))
                    elif la=="answer" and lb=="question": pairs.add((b,a)); stats["link_reversed"] += 1
                    else: stats["link_unorientable"] += 1
                else:
                    pairs.add((a,b))
        docs.append(dict(doc_id=stem, width=W, height=H, image=img_path,
                         entities=ents, pairs=sorted(pairs), n_unique_links=len(uniq)))
    return docs, stats, raw_entities, raw_links

TRAIN_DOCS, tr_stats, TR_RAW_E, TR_RAW_L = load_split("training_data")
TEST_DOCS,  te_stats, TE_RAW_E, TE_RAW_L = load_split("testing_data")

def describe(docs, name, stats, raw_e, raw_l):
    ents = [e for d in docs for e in d["entities"]]
    lab  = collections.Counter(e["label"] for e in ents)
    words= sum(len(e["words"]) for e in ents)
    uniq = sum(d["n_unique_links"] for d in docs)
    pairs= sum(len(d["pairs"]) for d in docs)
    print(f"\n{name}: {len(docs)} forms | {len(ents)} entities (raw {raw_e}) | {words} word slots")
    print(f"  labels: {dict(lab.most_common())}")
    print(f"  unique links {uniq} (raw {raw_l}) -> usable q->a pairs {pairs}")
    if stats: print(f"  notes: {dict(stats)}")
    return dict(forms=len(docs), entities=len(ents), words=words, uniq=uniq, pairs=pairs,
                labels=dict(lab), raw_e=raw_e, raw_l=raw_l)

TR = describe(TRAIN_DOCS, "TRAIN", tr_stats, TR_RAW_E, TR_RAW_L)
TE = describe(TEST_DOCS,  "TEST",  te_stats, TE_RAW_E, TE_RAW_L)

# Asserted on the RAW counts, before any modelling filter - the point is to prove this
# IS the published dataset, independently of preprocessing choices. (An earlier version
# asserted post-filter and failed, which is how the 65 blank fields were found.)
if CFG["assert_dataset_identity"]:
    exp = dict(train_forms=80, test_forms=20, train_entities=6549, test_entities=1435,
               train_links=1604, test_links=406)
    got = dict(train_forms=TR["forms"], test_forms=TE["forms"],
               train_entities=TR["raw_e"], test_entities=TE["raw_e"],
               train_links=TR["raw_l"], test_links=TE["raw_l"])
    bad = {k:(exp[k],got[k]) for k in exp if exp[k]!=got[k]}
    assert not bad, (f"SinFUND identity assert FAILED (expected, got): {bad}\n"
                     f"This is not the published dataset - results are not comparable.")
    print("\nIDENTITY ASSERT PASSED: 80/20 forms, 6549/1435 entities, 1604/406 relations")
    print("  (matches the published SinFUND counts exactly)")

# Validation carved from TRAIN only; the 20 test forms are never touched until Cell 9.
rng = random.Random(CFG["seed"])
_idx = list(range(len(TRAIN_DOCS))); rng.shuffle(_idx)
_n_val = max(1, int(len(_idx)*CFG["val_fraction"]))
VAL_DOCS   = [TRAIN_DOCS[i] for i in _idx[:_n_val]]
FIT_DOCS   = [TRAIN_DOCS[i] for i in _idx[_n_val:]]
assert not (set(d["doc_id"] for d in FIT_DOCS) & set(d["doc_id"] for d in VAL_DOCS))
print(f"\nsplit: fit {len(FIT_DOCS)} | val {len(VAL_DOCS)} | test {len(TEST_DOCS)} (untouched)")

PUBLISHED = dict(ser_f1=0.7560, ser_p=0.7552, ser_r=0.7562,
                 re_f1=0.3610,  re_p=0.2809,  re_r=0.5051)
print(f"\npublished SinFUND baseline to beat: SER F1 {PUBLISHED['ser_f1']}  "
      f"RE F1 {PUBLISHED['re_f1']} (P {PUBLISHED['re_p']} / R {PUBLISHED['re_r']})")


SinFUND found at: /kaggle/input/datasets/danushamsc25/sinfund/dataset
  (note: 2 candidates, using the first)

TRAIN: 80 forms | 6549 entities (raw 6549) | 17427 word slots
  labels: {'other': 3070, 'question': 1628, 'answer': 1614, 'header': 237}
  unique links 1604 (raw 1604) -> usable q->a pairs 1588
  notes: {'link_reversed': 162, 'blank_field_kept': 65, 'link_unorientable': 16}

TEST: 20 forms | 1435 entities (raw 1435) | 4047 word slots
  labels: {'other': 623, 'answer': 422, 'question': 349, 'header': 41}
  unique links 406 (raw 406) -> usable q->a pairs 401
  notes: {'link_reversed': 59, 'link_unorientable': 5}

IDENTITY ASSERT PASSED: 80/20 forms, 6549/1435 entities, 1604/406 relations
  (matches the published SinFUND counts exactly)

split: fit 68 | val 12 | test 20 (untouched)

published SinFUND baseline to beat: SER F1 0.756  RE F1 0.361 (P 0.2809 / R 0.5051)


In [17]:
# ============================================================================
# Cell 3 - Tokenizer, Sinhala fertility measurement, IOB label space
# ============================================================================
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(ARM["repo"])
print(f"tokenizer: {tok.__class__.__name__}  vocab {len(tok)}")

# Blank answer fields (Cell 2) need a surface form that yields exactly one token.
# unk_token is in the vocabulary by construction, so it always does.
BLANK_SURFACE = tok.unk_token or "<unk>"
assert len(tok.tokenize(BLANK_SURFACE)) >= 1
print(f"blank-field sentinel: {BLANK_SURFACE!r} -> {tok.tokenize(BLANK_SURFACE)}")

# --- Sinhala fertility -------------------------------------------------------
# Published claim: GPT-2-style pre-tokenisation (`\p{L}+`) excludes the Unicode Mark
# category and shatters abugidas at every vowel sign - Sinhala 4.12 tokens/word vs 1.27
# with a mark-aware class. XLM-R uses SentencePiece Unigram, NOT ByteLevel BPE, so it
# may be unaffected. That is a hypothesis; measure it rather than assume either way.
def is_sinhala(s): return any(0x0D80 <= ord(c) <= 0x0DFF for c in s)

_sin_words, _lat_words = [], []
for d in TRAIN_DOCS:
    for e in d["entities"]:
        for w in e["words"]:
            t = w["text"]
            if not t: continue
            (_sin_words if is_sinhala(t) else _lat_words).append(t)

def fertility(words):
    if not words: return 0.0, 0, 0
    n_tok = sum(len(tok.tokenize(w)) for w in words)
    return n_tok/len(words), n_tok, len(words)

sin_f, sin_t, sin_n = fertility(_sin_words)
lat_f, lat_t, lat_n = fertility(_lat_words)
unk_id = tok.unk_token_id
sin_unk = sum(1 for w in _sin_words for i in tok.encode(w, add_special_tokens=False) if i == unk_id)
print(f"\n--- tokenizer fertility on real SinFUND text ---")
print(f"  Sinhala : {sin_f:.2f} tokens/word   ({sin_t} tokens / {sin_n} words)")
print(f"  Latin   : {lat_f:.2f} tokens/word   ({lat_t} tokens / {lat_n} words)")
print(f"  Sinhala subwords decoding to <unk>: {sin_unk} ({sin_unk/max(1,sin_t):.2%})")
print(f"  reference: the abugida-shatter paper reports 4.12 broken vs 1.27 fixed.")
print(f"  >>> {'HIGH - sequence budget is affected' if sin_f > 3.0 else 'OK - no shatter pathology'}")
if sin_unk / max(1, sin_t) > 0.01:
    print("  >>> WARNING: non-trivial <unk> rate. This tokenizer is losing Sinhala.")

# --- IOB label space ---------------------------------------------------------
ENT_LABELS = ["question","answer","header"]          # "other" maps to O
IOB = ["O"] + [f"{p}-{l.upper()}" for l in ENT_LABELS for p in ("B","I")]
L2I = {l:i for i,l in enumerate(IOB)}
I2L = {i:l for l,i in L2I.items()}
print(f"\nIOB labels ({len(IOB)}): {IOB}")


tokenizer: TokenizersBackend  vocab 250002
blank-field sentinel: '<unk>' -> ['<unk>']

--- tokenizer fertility on real SinFUND text ---
  Sinhala : 1.87 tokens/word   (21713 tokens / 11599 words)
  Latin   : 1.57 tokens/word   (9045 tokens / 5763 words)
  Sinhala subwords decoding to <unk>: 0 (0.00%)
  reference: the abugida-shatter paper reports 4.12 broken vs 1.27 fixed.
  >>> OK - no shatter pathology

IOB labels (7): ['O', 'B-QUESTION', 'I-QUESTION', 'B-ANSWER', 'I-ANSWER', 'B-HEADER', 'I-HEADER']


In [18]:
# ============================================================================
# Cell 4 - Encode documents into overlapping windows (words -> tokens/boxes/IOB)
# ============================================================================

# Measured: words/form median 213, max 366. After subword expansion this overflows the
# 512-token limit, so each document is encoded as overlapping windows and entity
# representations are pooled across them in Cell 5.
CLS_BOX, SEP_BOX, PAD_BOX = [0,0,0,0], [1000,1000,1000,1000], [0,0,0,0]

def reading_order(entities):
    # Sort top-to-bottom then left-to-right. The Donut track measured that feeding
    # entities in annotation order instead of reading order left 31.8% of adjacent
    # pairs inverted, which removed the sequential signal entirely.
    return sorted(entities, key=lambda e: (e["box"][1], e["box"][0]))

def encode_doc(doc):
    tok_ids, tok_boxes, tok_labels, tok_ent = [], [], [], []
    for e in reading_order(doc["entities"]):
        lab = e["label"]
        first = True
        for w in e["words"]:
            surface = w["text"] if w["text"] else BLANK_SURFACE
            ids = tok.encode(surface, add_special_tokens=False)
            if not ids: ids = [tok.unk_token_id]
            for k, i in enumerate(ids):
                tok_ids.append(i); tok_boxes.append(w["box"]); tok_ent.append(e["id"])
                if lab == "other":
                    tok_labels.append(L2I["O"])
                else:
                    pref = "B" if (first and k == 0) else "I"
                    tok_labels.append(L2I[f"{pref}-{lab.upper()}"])
            first = False
    return tok_ids, tok_boxes, tok_labels, tok_ent

def window_doc(doc, max_len=None, stride=None):
    max_len = max_len or CFG["max_length"]; stride = stride or CFG["window_stride"]
    ids, boxes, labels, ents = encode_doc(doc)
    body = max_len - 2                       # room for <s> ... </s>
    windows, start = [], 0
    n = len(ids)
    if n == 0: return []
    while True:
        end = min(start + body, n)
        w_ids  = [tok.cls_token_id] + ids[start:end]   + [tok.sep_token_id]
        w_box  = [CLS_BOX]          + boxes[start:end] + [SEP_BOX]
        w_lab  = [-100]             + labels[start:end]+ [-100]
        w_ent  = [-1]               + ents[start:end]  + [-1]
        w_gidx = [-1]               + list(range(start,end)) + [-1]
        pad = max_len - len(w_ids)
        attn = [1]*len(w_ids) + [0]*pad
        w_ids += [tok.pad_token_id]*pad; w_box += [PAD_BOX]*pad
        w_lab += [-100]*pad; w_ent += [-1]*pad; w_gidx += [-1]*pad
        windows.append(dict(input_ids=w_ids, bbox=w_box, attention_mask=attn,
                            labels=w_lab, ent_ids=w_ent, gidx=w_gidx))
        if end >= n: break
        start = end - stride
    return windows

ENC = {}
for split, docs in (("fit",FIT_DOCS), ("val",VAL_DOCS), ("test",TEST_DOCS)):
    ENC[split] = [dict(doc=d, windows=window_doc(d)) for d in docs]
    nw = [len(x["windows"]) for x in ENC[split]]
    ntok = [sum(sum(w["attention_mask"]) for w in x["windows"]) for x in ENC[split]]
    print(f"{split:5}: {len(docs)} docs | windows/doc median {int(np.median(nw))} max {max(nw)} "
          f"| tokens/doc median {int(np.median(ntok))} max {max(ntok)}")

# Every entity must be recoverable from the encoding, or its RE embedding is undefined.
_missing = 0
for split in ENC:
    for x in ENC[split]:
        seen = {i for w in x["windows"] for i in w["ent_ids"] if i >= 0}
        _missing += sum(1 for e in x["doc"]["entities"] if e["id"] not in seen)
print(f"\nentities with no token in any window: {_missing}")
assert _missing == 0, "some entities are unrecoverable - RE embeddings would be undefined"

_lab_count = collections.Counter(l for split in ENC for x in ENC[split]
                                 for w in x["windows"] for l in w["labels"] if l >= 0)
print("token-level IOB distribution:", {I2L[k]:v for k,v in sorted(_lab_count.items())})


fit  : 68 docs | windows/doc median 1 max 2 | tokens/doc median 372 max 746
val  : 12 docs | windows/doc median 1 max 2 | tokens/doc median 389 max 789
test : 20 docs | windows/doc median 1 max 2 | tokens/doc median 343 max 735

entities with no token in any window: 0
token-level IOB distribution: {'O': 13311, 'B-QUESTION': 2041, 'I-QUESTION': 11341, 'B-ANSWER': 2098, 'I-ANSWER': 7575, 'B-HEADER': 285, 'I-HEADER': 2709}


In [19]:
# ============================================================================
# Cell 5 - Models: SER token classifier + a FRESH bilinear relation head
# ============================================================================
import torch.nn as nn
from transformers import AutoModel, AutoModelForTokenClassification, AutoConfig

DEVICE = "cuda"

def model_inputs(batch, uses_bbox):
    out = dict(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
    if uses_bbox: out["bbox"] = batch["bbox"]
    return out

def _from_pretrained(cls, repo, **kw):
    """Not every architecture declares sdpa support; requesting it then raises
    ValueError at load time. Four Kaggle runs in a sibling track died on
    transformers-version incompatibilities of exactly this shape, so degrade
    gracefully and SAY SO rather than failing the run."""
    try:
        return cls.from_pretrained(repo, attn_implementation=ATTN_IMPL, **kw)
    except (ValueError, TypeError) as e:
        print(f"  [attn] {ATTN_IMPL} rejected by {cls.__name__} ({type(e).__name__}); "
              f"falling back to the default attention implementation")
        return cls.from_pretrained(repo, **kw)

def build_ser():
    m = _from_pretrained(AutoModelForTokenClassification, ARM["repo"],
                         num_labels=len(IOB), id2label=I2L, label2id=L2I)
    return m.to(DEVICE)

# The LiLT reference implementation's biaffine RE head was ported into a sibling
# notebook and hit FOUR separate transformers incompatibilities across as many Kaggle
# runs; the last recorded run still had RE_MODEL_OK=False. This is ~40 lines, has no
# version surface, and is ours to debug.
class BilinearRE(nn.Module):
    def __init__(self, hidden, proj=128, dropout=0.1):
        super().__init__()
        self.q = nn.Sequential(nn.Linear(hidden, proj), nn.GELU(), nn.Dropout(dropout))
        self.a = nn.Sequential(nn.Linear(hidden, proj), nn.GELU(), nn.Dropout(dropout))
        self.bil  = nn.Bilinear(proj, proj, 1)
        self.lin  = nn.Linear(2*proj, 1)        # additive term: a pure bilinear form
                                                 # cannot express single-entity priors
                                                 # (e.g. "this answer is already taken")
    def forward(self, q_emb, a_emb):
        q, a = self.q(q_emb), self.a(a_emb)
        return (self.bil(q, a) + self.lin(torch.cat([q, a], dim=-1))).squeeze(-1)

class REModel(nn.Module):
    def __init__(self, repo, proj, dropout):
        super().__init__()
        self.enc  = _from_pretrained(AutoModel, repo)
        self.head = BilinearRE(self.enc.config.hidden_size, proj, dropout)
    def encode_doc(self, windows, uses_bbox):
        """Mean-pool each entity's token states across ALL windows containing it."""
        sums, counts = {}, {}
        for w in windows:
            batch = {k: torch.tensor([w[k]], device=DEVICE) for k in ("input_ids","attention_mask")}
            if uses_bbox: batch["bbox"] = torch.tensor([w["bbox"]], device=DEVICE)
            hs = self.enc(**model_inputs(batch, uses_bbox)).last_hidden_state[0]
            eids = torch.tensor(w["ent_ids"], device=DEVICE)
            for eid in eids.unique().tolist():
                if eid < 0: continue
                v = hs[eids == eid].sum(0)
                sums[eid]   = sums.get(eid, 0) + v
                counts[eid] = counts.get(eid, 0) + int((eids == eid).sum())
        return {k: sums[k]/counts[k] for k in sums}

def candidate_pairs(doc):
    """All question x answer pairs in a document. Measured: 37,709 train candidates vs
    1,604 gold positives = 4.3%, so the loss must be class-weighted."""
    qs = [e["id"] for e in doc["entities"] if e["label"] == "question"]
    as_ = [e["id"] for e in doc["entities"] if e["label"] == "answer"]
    return qs, as_

_n_cand = sum(len(candidate_pairs(x["doc"])[0]) * len(candidate_pairs(x["doc"])[1])
              for x in ENC["fit"])
_n_pos  = sum(len(x["doc"]["pairs"]) for x in ENC["fit"])
POS_WEIGHT = CFG["re_pos_weight"] or max(1.0, (_n_cand - _n_pos) / max(1, _n_pos))
print(f"RE candidates (fit): {_n_cand} | positives {_n_pos} ({_n_pos/max(1,_n_cand):.2%})")
print(f"pos_weight = {POS_WEIGHT:.1f}")
print(f"\nSER labels: {len(IOB)} | RE proj dim: {CFG['re_proj_dim']}")


RE candidates (fit): 30892 | positives 1340 (4.34%)
pos_weight = 22.1

SER labels: 7 | RE proj dim: 128


In [20]:
# ============================================================================
# Cell 6 - Smoke test + gold-everything sanity tier
# ============================================================================

# --- metrics -----------------------------------------------------------------
def ser_spans(labels):
    """IOB token labels -> set of (start, end, type) spans."""
    spans, cur = set(), None
    for i, l in enumerate(labels):
        if l is None or l < 0: continue
        tag = I2L[l]
        if tag == "O":
            if cur: spans.add(cur); cur = None
        else:
            pre, typ = tag.split("-", 1)
            if pre == "B" or cur is None or cur[2] != typ:
                if cur: spans.add(cur)
                cur = (i, i, typ)
            else:
                cur = (cur[0], i, cur[2])
    if cur: spans.add(cur)
    return spans

def prf(pred, gold):
    tp = len(pred & gold)
    p = tp/len(pred) if pred else 0.0
    r = tp/len(gold) if gold else 0.0
    f = 2*p*r/(p+r) if (p+r) else 0.0
    return dict(precision=p, recall=r, f1=f, tp=tp, n_pred=len(pred), n_gold=len(gold))

def ser_score(all_pred, all_gold):
    """Entity-level (span) P/R/F1, overall and per class."""
    P = set(); G = set()
    for di,(pl,gl) in enumerate(zip(all_pred, all_gold)):
        P |= {(di,)+s for s in ser_spans(pl)}
        G |= {(di,)+s for s in ser_spans(gl)}
    out = dict(overall=prf(P, G))
    for t in (l.upper() for l in ENT_LABELS):
        out[t] = prf({x for x in P if x[3]==t}, {x for x in G if x[3]==t})
    return out

def re_score(pred_pairs, gold_pairs):
    """Entity-PAIR F1. Rule 14: never score RE with entity-level F1 - that is
    'SER dressed as RE' (KIEval)."""
    return prf(set(pred_pairs), set(gold_pairs))

# --- sanity tier: gold in, gold out, must be exactly 1.0 ---------------------
_gold_lab = [[l for l in w["labels"]] for x in ENC["val"] for w in x["windows"]]
_s = ser_score(_gold_lab, _gold_lab)
_gp = [(x["doc"]["doc_id"],a,b) for x in ENC["val"] for (a,b) in x["doc"]["pairs"]]
_r = re_score(_gp, _gp)
print(f"SANITY TIER (gold->gold):  SER F1 {_s['overall']['f1']:.4f}   RE F1 {_r['f1']:.4f}")
assert abs(_s["overall"]["f1"]-1.0) < 1e-9 and abs(_r["f1"]-1.0) < 1e-9, (
    "harness is broken - gold-vs-gold must score exactly 1.0; no other number counts")
print("  both exactly 1.0000 - harness is sound")

# --- forward/backward smoke test ---------------------------------------------
if True:
    _m = build_ser()
    _w = ENC["fit"][0]["windows"][0]
    _b = {k: torch.tensor([_w[k]], device=DEVICE) for k in ("input_ids","attention_mask","bbox","labels")}
    _m.train()
    with torch.autocast("cuda", dtype=AMP_DTYPE):
        _out = _m(**model_inputs(_b, ARM["uses_bbox"]), labels=_b["labels"])
    print(f"\nSER smoke: loss {_out.loss.item():.4f}  logits {tuple(_out.logits.shape)}")
    assert torch.isfinite(_out.loss), "SER loss is NaN/Inf on the first batch"
    _out.loss.backward()
    _ng = sum(1 for p in _m.parameters() if p.grad is not None and p.grad.abs().sum() > 0)
    print(f"  params with non-zero grad: {_ng}")
    assert _ng > 0
    del _m, _out; gc.collect(); torch.cuda.empty_cache()

    _re = REModel(ARM["repo"], CFG["re_proj_dim"], CFG["re_dropout"]).to(DEVICE)
    _x = ENC["fit"][0]
    with torch.autocast("cuda", dtype=AMP_DTYPE):
        _emb = _re.encode_doc(_x["windows"], ARM["uses_bbox"])
        _qs, _as = candidate_pairs(_x["doc"])
        if _qs and _as:
            _qe = torch.stack([_emb[i] for i in _qs]).repeat_interleave(len(_as), 0)
            _ae = torch.stack([_emb[i] for i in _as]).repeat(len(_qs), 1)
            _lg = _re.head(_qe, _ae)
            print(f"RE smoke: {len(_qs)}Q x {len(_as)}A = {_lg.numel()} candidate logits, "
                  f"finite={bool(torch.isfinite(_lg).all())}")
            assert torch.isfinite(_lg).all()
    print(f"  entity embeddings recovered: {len(_emb)}/{len(_x['doc']['entities'])}")
    assert len(_emb) == len(_x["doc"]["entities"])
    print(f"peak VRAM: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")
    del _re; gc.collect(); torch.cuda.empty_cache()
print("\nSMOKE TEST PASSED")


SANITY TIER (gold->gold):  SER F1 1.0000   RE F1 1.0000
  both exactly 1.0000 - harness is sound
  [attn] sdpa rejected by AutoModelForTokenClassification (ValueError); falling back to the default attention implementation


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

LiltForTokenClassification LOAD REPORT from: SCUT-DLVCLab/lilt-infoxlm-base
Key                     | Status     | 
------------------------+------------+-
embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight     | UNEXPECTED | 
pooler.dense.bias       | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



SER smoke: loss 1.9883  logits (1, 512, 7)
  params with non-zero grad: 388
  [attn] sdpa rejected by AutoModel (ValueError); falling back to the default attention implementation


Loading weights:   0%|          | 0/400 [00:00<?, ?it/s]

LiltModel LOAD REPORT from: SCUT-DLVCLab/lilt-infoxlm-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RE smoke: 15Q x 16A = 240 candidate logits, finite=True
  entity embeddings recovered: 54/54
peak VRAM: 10.20 GB

SMOKE TEST PASSED


In [21]:
# ============================================================================
# Cell 7 - Train SER (entity-level span F1, per-class reported)
# ============================================================================
from torch.utils.data import Dataset, DataLoader
from transformers import get_linear_schedule_with_warmup

class WindowDS(Dataset):
    def __init__(self, enc): self.w = [w for x in enc for w in x["windows"]]
    def __len__(self): return len(self.w)
    def __getitem__(self, i):
        w = self.w[i]
        return {k: torch.tensor(w[k]) for k in ("input_ids","attention_mask","bbox","labels")}

@torch.no_grad()
def ser_predict_docs(model, enc):
    """Assemble document-level predictions, averaging logits over overlapping windows."""
    model.eval()
    preds, golds = [], []
    for x in enc:
        n = max((g for w in x["windows"] for g in w["gidx"]), default=-1) + 1
        if n <= 0: preds.append([]); golds.append([]); continue
        acc = torch.zeros(n, len(IOB)); cnt = torch.zeros(n); gold = [-100]*n
        for w in x["windows"]:
            b = {k: torch.tensor([w[k]], device=DEVICE) for k in ("input_ids","attention_mask","bbox")}
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                lg = model(**model_inputs(b, ARM["uses_bbox"])).logits[0].float().cpu()
            for pos, g in enumerate(w["gidx"]):
                if g < 0: continue
                acc[g] += lg[pos]; cnt[g] += 1; gold[g] = w["labels"][pos]
        keep = cnt > 0
        pred = acc[keep].argmax(-1).tolist()
        preds.append(pred); golds.append([g for g,k in zip(gold, keep.tolist()) if k])
    return preds, golds

def run_ser():
    model = build_ser()
    dl = DataLoader(WindowDS(ENC["fit"]), batch_size=CFG["ser_batch"], shuffle=True)
    steps = len(dl)*CFG["ser_epochs"]
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["ser_lr"], weight_decay=CFG["weight_decay"])
    sch = get_linear_schedule_with_warmup(opt, int(steps*CFG["warmup_ratio"]), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE==torch.float16))
    best, best_state, hist = -1.0, None, []
    t0 = time.time(); deadline = t0 + CFG["train_hours_budget"]*3600*0.5   # SER gets half the budget
    print(f"SER: {len(dl)} steps/epoch x {CFG['ser_epochs']} epochs = {steps} steps")
    for ep in range(CFG["ser_epochs"]):
        model.train()
        tot = 0.0
        for b in dl:
            b = {k: v.to(DEVICE) for k,v in b.items()}
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                loss = model(**model_inputs(b, ARM["uses_bbox"]), labels=b["labels"]).loss
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["max_grad_norm"])
            scaler.step(opt); scaler.update(); sch.step()
            tot += loss.item()
        if (ep+1) % 5 == 0 or ep == CFG["ser_epochs"]-1:
            pr, gl = ser_predict_docs(model, ENC["val"])
            s = ser_score(pr, gl)
            hist.append(dict(epoch=ep+1, loss=tot/len(dl), val_f1=s["overall"]["f1"]))
            print(f"  ep {ep+1:3}: loss {tot/len(dl):.4f}  val span F1 {s['overall']['f1']:.4f}  "
                  f"(Q {s['QUESTION']['f1']:.3f} A {s['ANSWER']['f1']:.3f} H {s['HEADER']['f1']:.3f})")
            if s["overall"]["f1"] > best:
                best = s["overall"]["f1"]
                best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
                print(f"       new best -> {best:.4f}")
        if time.time() > deadline:
            print(f"  [budget] SER half-budget reached at epoch {ep+1}; stopping."); break
    if best_state: model.load_state_dict(best_state); print(f"restored best SER (val F1 {best:.4f})")
    return model, hist, best

SER_MODEL = SER_HIST = None; SER_BEST = float("nan")
if CFG["do_ser"]:
    _t = time.time()
    SER_MODEL, SER_HIST, SER_BEST = run_ser()
    print(f"\nSER done in {(time.time()-_t)/60:.1f} min; best val span F1 {SER_BEST:.4f}")
    print(f"peak VRAM: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")
else:
    print("do_ser=False -> skipping")


  [attn] sdpa rejected by AutoModelForTokenClassification (ValueError); falling back to the default attention implementation


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

LiltForTokenClassification LOAD REPORT from: SCUT-DLVCLab/lilt-infoxlm-base
Key                     | Status     | 
------------------------+------------+-
embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight     | UNEXPECTED | 
pooler.dense.bias       | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


SER: 38 steps/epoch x 2 epochs = 76 steps


/tmp/ipykernel_58/2358779505.py:56: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sch.step()


  ep   2: loss 1.5481  val span F1 0.0483  (Q 0.017 A 0.120 H 0.000)
       new best -> 0.0483
restored best SER (val F1 0.0483)

SER done in 0.4 min; best val span F1 0.0483
peak VRAM: 10.20 GB


In [22]:
# ============================================================================
# Cell 8 - Train the relation head (entity-pair F1, threshold tuned on val)
# ============================================================================

def re_forward_doc(model, x, uses_bbox):
    emb = model.encode_doc(x["windows"], uses_bbox)
    qs, as_ = candidate_pairs(x["doc"])
    if not qs or not as_: return None
    qe = torch.stack([emb[i] for i in qs]).repeat_interleave(len(as_), 0)
    ae = torch.stack([emb[i] for i in as_]).repeat(len(qs), 1)
    logits = model.head(qe, ae)
    gold = torch.zeros(len(qs)*len(as_), device=DEVICE)
    gp = set(x["doc"]["pairs"])
    for i,q in enumerate(qs):
        for j,a in enumerate(as_):
            if (q,a) in gp: gold[i*len(as_)+j] = 1.0
    return logits, gold, qs, as_

@torch.no_grad()
def re_eval(model, enc, thresholds=(0.3,0.4,0.5,0.6,0.7)):
    model.eval()
    scored, gold_all = [], []
    for x in enc:
        out = re_forward_doc(model, x, ARM["uses_bbox"])
        if out is None: continue
        logits, _, qs, as_ = out
        pr = torch.sigmoid(logits.float()).cpu()
        did = x["doc"]["doc_id"]
        for i,q in enumerate(qs):
            for j,a in enumerate(as_):
                scored.append((pr[i*len(as_)+j].item(), (did,q,a)))
        gold_all += [(did,a,b) for (a,b) in x["doc"]["pairs"]]
    best = dict(f1=-1)
    for t in thresholds:
        m = re_score([k for s,k in scored if s >= t], gold_all)
        m["threshold"] = t
        if m["f1"] > best["f1"]: best = m
    return best, scored, gold_all

def run_re():
    model = REModel(ARM["repo"], CFG["re_proj_dim"], CFG["re_dropout"]).to(DEVICE)
    params = list(model.parameters())
    opt = torch.optim.AdamW(params, lr=CFG["re_lr"], weight_decay=CFG["weight_decay"])
    steps = len(ENC["fit"])*CFG["re_epochs"]
    sch = get_linear_schedule_with_warmup(opt, int(steps*CFG["warmup_ratio"]), steps)
    lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT, device=DEVICE))
    best, best_state, hist = -1.0, None, []
    t0 = time.time(); deadline = t0 + CFG["train_hours_budget"]*3600*0.5
    print(f"RE: {len(ENC['fit'])} docs x {CFG['re_epochs']} epochs, pos_weight {POS_WEIGHT:.1f}")
    order = list(range(len(ENC["fit"])))
    for ep in range(CFG["re_epochs"]):
        model.train(); random.shuffle(order); tot, nb_ = 0.0, 0
        for i in order:
            out = re_forward_doc(model, ENC["fit"][i], ARM["uses_bbox"])
            if out is None: continue
            logits, gold, _, _ = out
            loss = lossf(logits.float(), gold)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, CFG["max_grad_norm"])
            opt.step(); sch.step(); tot += loss.item(); nb_ += 1
        if (ep+1) % 5 == 0 or ep == CFG["re_epochs"]-1:
            m,_,_ = re_eval(model, ENC["val"])
            hist.append(dict(epoch=ep+1, loss=tot/max(1,nb_), val_f1=m["f1"], thr=m["threshold"]))
            print(f"  ep {ep+1:3}: loss {tot/max(1,nb_):.4f}  val pair F1 {m['f1']:.4f} "
                  f"(P {m['precision']:.3f} R {m['recall']:.3f} @thr {m['threshold']})")
            if m["f1"] > best:
                best = m["f1"]
                best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
                print(f"       new best -> {best:.4f}")
        if time.time() > deadline:
            print(f"  [budget] RE half-budget reached at epoch {ep+1}; stopping."); break
    if best_state: model.load_state_dict(best_state); print(f"restored best RE (val pair F1 {best:.4f})")
    m,_,_ = re_eval(model, ENC["val"])
    return model, hist, best, m["threshold"]

RE_MODEL = RE_HIST = None; RE_BEST = float("nan"); RE_THR = 0.5
if CFG["do_re"]:
    _t = time.time()
    RE_MODEL, RE_HIST, RE_BEST, RE_THR = run_re()
    print(f"\nRE done in {(time.time()-_t)/60:.1f} min; best val pair F1 {RE_BEST:.4f} "
          f"@ threshold {RE_THR}")
    print(f"peak VRAM: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")
else:
    print("do_re=False -> skipping")


  [attn] sdpa rejected by AutoModel (ValueError); falling back to the default attention implementation


Loading weights:   0%|          | 0/400 [00:00<?, ?it/s]

LiltModel LOAD REPORT from: SCUT-DLVCLab/lilt-infoxlm-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RE: 68 docs x 2 epochs, pos_weight 22.1
  ep   2: loss 1.5197  val pair F1 0.0702 (P 0.036 R 1.000 @thr 0.3)
       new best -> 0.0702
restored best RE (val pair F1 0.0702)

RE done in 0.7 min; best val pair F1 0.0702 @ threshold 0.3
peak VRAM: 10.20 GB


In [23]:
# ============================================================================
# Cell 9 - Final evaluation on the held-out 20 test forms + comparison
# ============================================================================
RESULTS = dict(notebook_version=NOTEBOOK_VERSION, arm=CFG["arm"], repo=ARM["repo"],
               config={k:str(v) for k,v in CFG.items()},
               sinhala_fertility=sin_f, published=PUBLISHED)

print("="*78)
print(f"FINAL TEST EVALUATION - {len(TEST_DOCS)} held-out SinFUND forms")
print(f"arm: {CFG['arm']} ({ARM['repo']}), layout stream: {ARM['uses_bbox']}")
print("="*78)

if SER_MODEL is not None:
    pr, gl = ser_predict_docs(SER_MODEL, ENC["test"])
    S = ser_score(pr, gl)
    RESULTS["ser"] = {k:{kk:vv for kk,vv in v.items()} for k,v in S.items()}
    print(f"\nSER (entity-level span F1)")
    print(f"  {'class':10} {'P':>8} {'R':>8} {'F1':>8}   {'pred':>6} {'gold':>6}")
    for k in ("overall",)+tuple(l.upper() for l in ENT_LABELS):
        v = S[k]
        print(f"  {k:10} {v['precision']:8.4f} {v['recall']:8.4f} {v['f1']:8.4f}   "
              f"{v['n_pred']:6} {v['n_gold']:6}")
    d = S["overall"]["f1"] - PUBLISHED["ser_f1"]
    print(f"\n  published SER F1 {PUBLISHED['ser_f1']:.4f} -> ours {S['overall']['f1']:.4f}  "
          f"({'+' if d>=0 else ''}{d:.4f})")
    print(f"  NOTE: the published number is measured GIVEN GOLD OCR TEXT. This run is also")
    print(f"        given gold text (SinFUND transcriptions), so the settings match.")

if RE_MODEL is not None:
    M, scored, gold_all = re_eval(RE_MODEL, ENC["test"], thresholds=(RE_THR,))
    RESULTS["re"] = M
    print(f"\nRE (entity-PAIR F1, threshold {M['threshold']} tuned on val)")
    print(f"  precision {M['precision']:.4f}   recall {M['recall']:.4f}   F1 {M['f1']:.4f}")
    print(f"  predicted {M['n_pred']} pairs | gold {M['n_gold']} | correct {M['tp']}")
    d = M["f1"] - PUBLISHED["re_f1"]
    print(f"\n  published RE F1 {PUBLISHED['re_f1']:.4f} (P {PUBLISHED['re_p']:.4f} "
          f"R {PUBLISHED['re_r']:.4f}) -> ours {M['f1']:.4f}  ({'+' if d>=0 else ''}{d:.4f})")

print("\n" + "="*78)
print("SCOREBOARD - SinFUND test split")
print("="*78)
print(f"  {'system':46} {'SER F1':>9} {'RE F1':>9}")
print("  " + "-"*66)
print(f"  {'SinFUND paper (LiLT+XLM-R, Sinhala-only)':46} "
      f"{PUBLISHED['ser_f1']:9.4f} {PUBLISHED['re_f1']:9.4f}")
_s = RESULTS.get("ser",{}).get("overall",{}).get("f1", float('nan'))
_r = RESULTS.get("re",{}).get("f1", float('nan'))
print(f"  {'THIS RUN (' + CFG['arm'] + ', Sinhala-only)':46} {_s:9.4f} {_r:9.4f}")
print("  " + "-"*66)
print("  reference points (different datasets - NOT comparable, for context only):")
print(f"  {'  KH-FUNSD Khmer, LayoutLMv3 (158 docs)':46} {0.653:9.4f} {float('nan'):>9}")
print(f"  {'  FUNSD English, LiLT[InfoXLM] lang-specific':46} {0.8251:9.4f} {0.6781:9.4f}")
print(f"  {'  FUNSD English, LiLT[InfoXLM] multitask':46} {0.8585:9.4f} {0.8125:9.4f}")

print("\nRoadmap: this is Step 1 (Sinhala-only control). Step 2 - multitask on")
print("FUNSD+XFUND before SinFUND - is the +13 RE F1 lever. See GuideForFormUnderstanding.")

with open(os.path.join(RUN_DIR,"RESULTS.json"),"w") as f:
    json.dump(RESULTS, f, indent=2, default=str)
print(f"\nsaved: {os.path.join(RUN_DIR,'RESULTS.json')}")


FINAL TEST EVALUATION - 20 held-out SinFUND forms
arm: lilt-infoxlm (SCUT-DLVCLab/lilt-infoxlm-base), layout stream: True

SER (entity-level span F1)
  class             P        R       F1     pred   gold
  overall      0.0340   0.0505   0.0407     1205    812
  QUESTION     0.0202   0.0544   0.0295      941    349
  ANSWER       0.0833   0.0521   0.0641      264    422
  HEADER       0.0000   0.0000   0.0000        0     41

  published SER F1 0.7560 -> ours 0.0407  (-0.7153)
  NOTE: the published number is measured GIVEN GOLD OCR TEXT. This run is also
        given gold text (SinFUND transcriptions), so the settings match.

RE (entity-PAIR F1, threshold 0.3 tuned on val)
  precision 0.0482   recall 1.0000   F1 0.0921
  predicted 8311 pairs | gold 401 | correct 401

  published RE F1 0.3610 (P 0.2809 R 0.5051) -> ours 0.0921  (-0.2689)

SCOREBOARD - SinFUND test split
  system                                            SER F1     RE F1
  ---------------------------------------------

## Reading this run's output

**In order — stop at the first thing that is wrong.**

1. **Cell 1 revision banner** — is it the revision you meant to run? A sibling track lost a
   whole session evaluating a stale notebook.
2. **Cell 2 identity assert** — must print `80/20 forms, 6549/1435 entities, 1604/406
   relations`. If it fails, this is not the published dataset and no number here is comparable.
3. **Cell 3 fertility** — settles an open question. `> 3.0 tokens/word` means the abugida
   shatter problem is real here and the sequence budget needs rethinking. A non-trivial
   `<unk>` rate means the tokenizer is losing Sinhala outright (that killed the Donut track).
4. **Cell 6 sanity tier** — gold-vs-gold must be **exactly 1.0000** for both SER and RE. It
   asserts, so a pass is silent-safe. A sibling notebook once scored an unearned 1.0000 because
   its pair builder ignored the labels it was handed — this tier exists to catch that class of bug.
5. **Cell 7 per-class SER** — `header` is 3.6% of entities. Expect it to be the worst class;
   KH-FUNSD (Khmer) sees the identical ordering (Q .783 / A .759 / **H .571** / Other .500).
6. **Cell 8 RE P/R split** — the published baseline is **P 0.2809 / R 0.5051**: it over-predicts.
   If ours shows the same shape, the relation head is data-starved the same way and Step 2
   (multitask pre-training of that head) is the indicated fix.

### What would make this run a result
Reproducing roughly **SER 0.7560 / RE 0.3610** on the `lilt-infoxlm` arm. That is the *control* —
it proves the harness matches the paper's setting. Only then does Step 2's multitask number mean
anything.

The `xlmr-textonly` arm is a genuinely novel data point regardless: it isolates how much the
layout modality contributes, which the paper never reported.

### What NOT to conclude
- Do not compare these numbers to the Donut track's F1 0.0203. That system reads raw pixels;
  this one is given gold text. Different task difficulty — see `rules.md` rule 15.
- Do not quote an improvement over "the paper's best setting" until the missing Table 9
  multilingual numbers are in hand. Against the **published Sinhala-only** baseline is fair.
